In [14]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [15]:
def avaliar_modelo_em_teste(caminho_csv_teste, caminho_modelo="models/best_regression_model.pkl"):
    print(f"Carregando modelo de: {caminho_modelo}...")
    try:
        model = joblib.load(caminho_modelo)
    except FileNotFoundError:
        print("Erro: Arquivo do modelo não encontrado. Certifique-se de que o arquivo .pkl está no mesmo diretório.")
        return

    print(f"Carregando dados de teste de: {caminho_csv_teste}...")
    try:
        df_test = pd.read_csv(caminho_csv_teste)
    except FileNotFoundError:
        print("Erro: Arquivo CSV de teste não encontrado.")
        return

    target = "Preco"

    categorical_features = [
        "Fabricante",
        "Modelo",
        "Categoria",
        "Combustivel",
        "Tipo_cambio",
        "Tração"
    ]

    numeric_features = [
        "Ano",
        "Volume_motor",
        "Km",
        "Airbags",
        "Idade"
    ]

    cols_necessarias = categorical_features + numeric_features

    # Verificar se todas as colunas necessárias existem
    if not all(col in df_test.columns for col in cols_necessarias):
        print("Erro: O dataset de teste não possui todas as colunas necessárias.")
        return

    # Caso não tenha Preco, gera apenas predições
    if target not in df_test.columns:
        print("Aviso: Coluna de preço (target) não encontrada. Apenas predições serão geradas, sem métricas.")

        X_test = df_test[cols_necessarias]

        y_pred = model.predict(X_test)

        df_test["Preco_Predito"] = y_pred

        print("\nPrimeiras 5 predições:")
        print(df_test[["Fabricante", "Modelo", "Preco_Predito"]].head())

        df_test.to_csv("predicoes_resultados.csv", index=False)
        print("\nResultados salvos em 'predicoes_resultados.csv'.")

    else:
        # Aplicar filtro de outliers pelo Preco
        q_low = df_test[target].quantile(0.01)
        q_hi = df_test[target].quantile(0.95)

        df_filtered = df_test[
            (df_test[target] > q_low) &
            (df_test[target] < q_hi)
        ].copy()

        print("\nFiltro aplicado no Preco:")
        print(f"Linhas antes do filtro: {len(df_test)}")
        print(f"Linhas depois do filtro: {len(df_filtered)}")
        print(f"Limite inferior: {q_low:.2f}")
        print(f"Limite superior: {q_hi:.2f}")

        X_test = df_filtered[cols_necessarias]
        y_true = df_filtered[target]

        print("\nRealizando predições e calculando métricas...")
        y_pred = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        print("\n" + "=" * 30)
        print("MÉTRICAS DE DESEMPENHO NO TESTE")
        print("=" * 30)
        print(f"R² Score: {r2:.4f}")
        print(f"RMSE:     {rmse:.2f}")
        print(f"MAE:      {mae:.2f}")
        print("=" * 30)

        df_filtered["Preco_Predito"] = y_pred
        df_filtered["Erro"] = df_filtered[target] - df_filtered["Preco_Predito"]
        df_filtered["Erro_Absoluto"] = abs(df_filtered["Erro"])

        df_filtered.to_csv("teste_com_predicoes_filtrado.csv", index=False)

        print("\nArquivo 'teste_com_predicoes_filtrado.csv' gerado com os valores reais e preditos.")

In [16]:
if __name__ == "__main__":
    # Substitua 'seu_arquivo_de_teste.csv' pelo nome do seu arquivo real
    # Para este exemplo, vamos testar com o próprio arquivo que temos
    avaliar_modelo_em_teste("database/test_tratado.csv")

Carregando modelo de: models/best_regression_model.pkl...
Carregando dados de teste de: database/test_tratado.csv...

Filtro aplicado no Preco:
Linhas antes do filtro: 8261
Linhas depois do filtro: 7755
Limite inferior: 47.00
Limite superior: 49858.00

Realizando predições e calculando métricas...

MÉTRICAS DE DESEMPENHO NO TESTE
R² Score: 0.6775
RMSE:     6796.30
MAE:      4444.50

Arquivo 'teste_com_predicoes_filtrado.csv' gerado com os valores reais e preditos.
